# Tâche 2 : Problème de classification binaire



## 1. Choix et description du dataset

### Dataset : Breast Cancer Wisconsin (Diagnostic)

**Contexte :**
- Domaine : Médecine / Oncologie
- Origine : Université du Wisconsin, données de diagnostic de cancer du sein
- Objectif : Prédire si une tumeur est bénigne (0) ou maligne (1) à partir de caractéristiques cellulaires

**Caractéristiques du dataset :**
- Nombre d'exemples : 569 observations
- Nombre de features : 30 variables prédictives (toutes numériques)
- Variable cible : `target` avec deux classes :
  - **0** : Maligne (Cancéreuse)
  - **1** : Bénigne (Non-cancéreuse)

**Features disponibles :**
Les features sont calculées à partir d'images numérisées de cellules, incluant :
- Rayon (radius) : distance moyenne du centre aux points du périmètre
- Texture (texture) : écart-type des valeurs de gris
- Périmètre (perimeter)
- Surface (area)
- Lissage (smoothness) : variation locale des longueurs de rayon
- Compacité (compactness) : périmètre² / surface - 1.0
- Concavité (concavity) : sévérité des portions concaves du contour
- Points concaves (concave points) : nombre de portions concaves
- Symétrie (symmetry)
- Dimension fractale (fractal dimension)

*Pour chaque caractéristique, on a la moyenne, l'erreur standard et la valeur "pire" (worst), donnant 30 features au total.*


## 2. Préparation des données


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from sklearn.tree import plot_tree
import warnings
warnings.filterwarnings('ignore')

# Configuration pour de meilleurs graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline


In [ ]:
# 2.1 Chargement du dataset
print("Chargement du dataset Breast Cancer...")
data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"\nDimensions du dataset : {df.shape}")
print(f"Nombre d'observations : {df.shape[0]}")
print(f"Nombre de features : {df.shape[1] - 1}")

# Affichage des premières lignes
print("\nAperçu des données (5 premières features) :")
df.iloc[:, :6].head()


In [ ]:
# Affichage des noms des features
print("Liste des features disponibles :")

for i, col in enumerate(df.columns[:-1], 1):
    print(f"{i:2d}. {col}")



In [ ]:
# 2.2 Exploration du dataset
print("Informations sur le dataset :")
print(df.info())


print("Statistiques descriptives :")
print(df.describe())


In [ ]:
# Vérification des valeurs manquantes
print("Valeurs manquantes par colonne :")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else " Aucune valeur manquante !")

print("\n" + "="*60)
# Distribution des classes
print("Distribution de la variable cible :")
print("-" * 60)
target_counts = df['target'].value_counts().sort_index()
print(f"Classe 0 (Maligne)  : {target_counts[0]} observations ({target_counts[0]/len(df)*100:.2f}%)")
print(f"Classe 1 (Bénigne)  : {target_counts[1]} observations ({target_counts[1]/len(df)*100:.2f}%)")
print(f"\nRatio : {target_counts[1]/target_counts[0]:.2f}:1 (Bénigne:Maligne)")
print("="*60)


In [ ]:
# Visualisation de la distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagramme en barres
axes[0].bar(['Maligne (0)', 'Bénigne (1)'], target_counts.values, 
            color=['#FF6B6B', '#4ECDC4'], edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Nombre d\'observations')
axes[0].set_title('Distribution des classes')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=12)

# Diagramme circulaire
colors = ['#FF6B6B', '#4ECDC4']
axes[1].pie(target_counts.values, labels=['Maligne', 'Bénigne'], 
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})
axes[1].set_title('Proportion des classes')

plt.tight_layout()
plt.show()


In [ ]:
# Visualisation de quelques features importantes
important_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 
                     'mean smoothness', 'mean compactness']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, feature in enumerate(important_features):
    for target_val in [0, 1]:
        data_subset = df[df['target'] == target_val][feature]
        axes[idx].hist(data_subset, alpha=0.6, bins=20, 
                      label=f"{'Maligne' if target_val == 0 else 'Bénigne'}", 
                      edgecolor='black')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Fréquence')
    axes[idx].set_title(f'Distribution de {feature}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Matrice de corrélation (features principales uniquement pour la lisibilité)
main_features = [col for col in df.columns if 'mean' in col]
correlation_subset = df[main_features + ['target']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_subset, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation (Features "mean" uniquement)')
plt.tight_layout()
plt.show()

print("\nCorrélation des features avec la variable cible (top 10) :")
target_corr = df.corr()['target'].sort_values(ascending=False)[1:11]
print(target_corr)


In [ ]:
# 2.3 Séparation des features (X) et de la cible (y)
X = df.drop('target', axis=1)
y = df['target']

print(f"Shape de X (features) : {X.shape}")
print(f"Shape de y (cible) : {y.shape}")
print(f"\nNombre de features : {X.shape[1]}")


In [ ]:
# 2.4 Division en train/test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille du jeu d'entraînement : {X_train.shape[0]} exemples")
print(f"Taille du jeu de test : {X_test.shape[0]} exemples")
print(f"\nProportion train/test : {X_train.shape[0]/len(X)*100:.1f}% / {X_test.shape[0]/len(X)*100:.1f}%")

# Vérifier la distribution des classes
print("\n" + "="*60)
print("Distribution des classes après split :")
print(f"Train - Classe 0: {sum(y_train==0)}, Classe 1: {sum(y_train==1)}")
print(f"Test  - Classe 0: {sum(y_test==0)}, Classe 1: {sum(y_test==1)}")
print("="*60)


In [ ]:
# 2.5 Standardisation des features
# Important pour la régression logistique qui est sensible à l'échelle des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standardisation effectuée avec succès !")
print(f"\nExemple - Moyenne avant standardisation (mean radius) : {X_train['mean radius'].mean():.4f}")
print(f"Exemple - Moyenne après standardisation : {X_train_scaled[:, 0].mean():.4f}")
print(f"Exemple - Écart-type après standardisation : {X_train_scaled[:, 0].std():.4f}")


## 3. Modèles à tester


### 3.1 Modèle 1 : Régression Logistique


In [ ]:
# Entraînement du modèle de régression logistique
print("Entraînement du modèle de Régression Logistique...")

logreg = LogisticRegression(max_iter=10000, random_state=42)
logreg.fit(X_train_scaled, y_train)

print("Modèle de Régression Logistique entraîné avec succès !")

# Prédictions
y_train_pred_logreg = logreg.predict(X_train_scaled)
y_test_pred_logreg = logreg.predict(X_test_scaled)


In [ ]:
# Évaluation de la Régression Logistique
print("="*70)
print("PERFORMANCES - RÉGRESSION LOGISTIQUE")
print("="*70)

# Métriques sur l'ensemble d'entraînement
accuracy_train_logreg = accuracy_score(y_train, y_train_pred_logreg)
precision_train_logreg = precision_score(y_train, y_train_pred_logreg)
recall_train_logreg = recall_score(y_train, y_train_pred_logreg)
f1_train_logreg = f1_score(y_train, y_train_pred_logreg)


print(f" Accuracy  : {accuracy_train_logreg:.4f} ({accuracy_train_logreg*100:.2f}%)")
print(f" Précision : {precision_train_logreg:.4f} ({precision_train_logreg*100:.2f}%)")
print(f" Rappel    : {recall_train_logreg:.4f} ({recall_train_logreg*100:.2f}%)")
print(f" F1-Score  : {f1_train_logreg:.4f}")

# Métriques sur l'ensemble de test
accuracy_test_logreg = accuracy_score(y_test, y_test_pred_logreg)
precision_test_logreg = precision_score(y_test, y_test_pred_logreg)
recall_test_logreg = recall_score(y_test, y_test_pred_logreg)
f1_test_logreg = f1_score(y_test, y_test_pred_logreg)


print(f" Accuracy  : {accuracy_test_logreg:.4f} ({accuracy_test_logreg*100:.2f}%)")
print(f" Précision : {precision_test_logreg:.4f} ({precision_test_logreg*100:.2f}%)")
print(f" Rappel    : {recall_test_logreg:.4f} ({recall_test_logreg*100:.2f}%)")
print(f" F1-Score  : {f1_test_logreg:.4f}")
print("="*70)


In [ ]:
# Rapport de classification détaillé
print("\nRapport de classification détaillé (Test Set) :")
print("-" * 70)
print(classification_report(y_test, y_test_pred_logreg, 
                          target_names=['Maligne (0)', 'Bénigne (1)']))


In [ ]:
# Matrice de confusion pour la Régression Logistique
cm_logreg = confusion_matrix(y_test, y_test_pred_logreg)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_logreg, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Maligne (0)', 'Bénigne (1)'],
            yticklabels=['Maligne (0)', 'Bénigne (1)'],
            cbar_kws={'label': 'Nombre de prédictions'})
plt.xlabel('Classe Prédite')
plt.ylabel('Classe Réelle')
plt.title('Matrice de Confusion - Régression Logistique')
plt.tight_layout()
plt.show()

print("\nMatrice de confusion (Test Set) :")
print(f"Vrais Négatifs(TN) : {cm_logreg[0, 0]}")
print(f"Faux Positifs (FP) : {cm_logreg[0, 1]}")
print(f"Faux Négatifs (FN) : {cm_logreg[1, 0]}")
print(f"Vrais Positifs(TP) : {cm_logreg[1, 1]}")


### 3.2 Modèle 2 : Arbre de Décision


In [ ]:
# Entraînement du modèle d'arbre de décision
print("Entraînement du modèle d'Arbre de Décision...")

# Utiliser max_depth pour éviter le sur-apprentissage
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)  # Pas besoin de standardisation pour les arbres

print("Modèle d'Arbre de Décision entraîné avec succès !")

# Prédictions
y_train_pred_tree = tree.predict(X_train)
y_test_pred_tree = tree.predict(X_test)


In [ ]:
# Évaluation de l'Arbre de Décision
print("="*70)
print("PERFORMANCES - ARBRE DE DÉCISION")
print("="*70)

# Métriques sur l'ensemble d'entraînement
accuracy_train_tree = accuracy_score(y_train, y_train_pred_tree)
precision_train_tree = precision_score(y_train, y_train_pred_tree)
recall_train_tree = recall_score(y_train, y_train_pred_tree)
f1_train_tree = f1_score(y_train, y_train_pred_tree)

print("\n Ensemble d'ENTRAÎNEMENT :")
print(f"Accuracy  : {accuracy_train_tree:.4f} ({accuracy_train_tree*100:.2f}%)")
print(f"Précision : {precision_train_tree:.4f} ({precision_train_tree*100:.2f}%)")
print(f"Rappel    : {recall_train_tree:.4f} ({recall_train_tree*100:.2f}%)")
print(f"F1-Score  : {f1_train_tree:.4f}")

# Métriques sur l'ensemble de test
accuracy_test_tree = accuracy_score(y_test, y_test_pred_tree)
precision_test_tree = precision_score(y_test, y_test_pred_tree)
recall_test_tree = recall_score(y_test, y_test_pred_tree)
f1_test_tree = f1_score(y_test, y_test_pred_tree)

print("\n Ensemble de TEST :")
print(f"Accuracy  : {accuracy_test_tree:.4f} ({accuracy_test_tree*100:.2f}%)")
print(f"Précision : {precision_test_tree:.4f} ({precision_test_tree*100:.2f}%)")
print(f"Rappel    : {recall_test_tree:.4f} ({recall_test_tree*100:.2f}%)")
print(f"F1-Score  : {f1_test_tree:.4f}")
print("="*70)


In [ ]:
# Rapport de classification détaillé
print("\nRapport de classification détaillé (Test Set) :")
print("-" * 70)
print(classification_report(y_test, y_test_pred_tree, 
                          target_names=['Maligne (0)', 'Bénigne (1)']))


In [ ]:
# Matrice de confusion pour l'Arbre de Décision
cm_tree = confusion_matrix(y_test, y_test_pred_tree)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Maligne (0)', 'Bénigne (1)'],
            yticklabels=['Maligne (0)', 'Bénigne (1)'],
            cbar_kws={'label': 'Nombre de prédictions'})
plt.xlabel('Classe Prédite')
plt.ylabel('Classe Réelle')
plt.title('Matrice de Confusion - Arbre de Décision')
plt.tight_layout()
plt.show()

print("\nMatrice de confusion (Test Set) :")
print(f"Vrais Négatifs(TN) : {cm_tree[0, 0]}")
print(f"Faux Positifs (FP) : {cm_tree[0, 1]}")
print(f"Faux Négatifs (FN) : {cm_tree[1, 0]}")
print(f"Vrais Positifs(TP) : {cm_tree[1, 1]}")


In [ ]:
# Visualisation de l'arbre de décision
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=X.columns, class_names=['Maligne', 'Bénigne'],
          filled=True, rounded=True, fontsize=10)
plt.title('Visualisation de l\'Arbre de Décision (max_depth=5)', fontsize=16)
plt.tight_layout()
plt.show()

# Importance des features
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': tree.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

print("\nTop 10 des features les plus importantes :")
print(feature_importance.to_string(index=False))


In [ ]:
# Visualisation de l'importance des features
plt.figure(figsize=(12, 6))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'])
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Top 15 Features les plus importantes (Arbre de Décision)')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 4. Comparaison et interprétation


### 4.1 Comparaison des métriques


In [ ]:
# Tableau comparatif des performances
comparison_df = pd.DataFrame({
    'Métrique': ['Accuracy', 'Précision', 'Rappel', 'F1-Score'],
    'Régression Logistique (Train)': [accuracy_train_logreg, precision_train_logreg, 
                                      recall_train_logreg, f1_train_logreg],
    'Régression Logistique (Test)': [accuracy_test_logreg, precision_test_logreg, 
                                     recall_test_logreg, f1_test_logreg],
    'Arbre de Décision (Train)': [accuracy_train_tree, precision_train_tree, 
                                   recall_train_tree, f1_train_tree],
    'Arbre de Décision (Test)': [accuracy_test_tree, precision_test_tree, 
                                  recall_test_tree, f1_test_tree]
})

print("="*90)
print("TABLEAU COMPARATIF DES PERFORMANCES")
print("="*90)
print(comparison_df.to_string(index=False))
print("="*90)


In [ ]:
# Visualisation comparative des métriques (Test Set)
metrics = ['Accuracy', 'Précision', 'Rappel', 'F1-Score']
logreg_scores = [accuracy_test_logreg, precision_test_logreg, recall_test_logreg, f1_test_logreg]
tree_scores = [accuracy_test_tree, precision_test_tree, recall_test_tree, f1_test_tree]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, logreg_scores, width, label='Régression Logistique', 
               color='steelblue', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width/2, tree_scores, width, label='Arbre de Décision', 
               color='seagreen', edgecolor='black', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Comparaison des Performances (Ensemble de Test)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Comparaison des matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Régression Logistique
sns.heatmap(cm_logreg, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Maligne (0)', 'Bénigne (1)'],
            yticklabels=['Maligne (0)', 'Bénigne (1)'],
            cbar_kws={'label': 'Nombre de prédictions'})
axes[0].set_xlabel('Classe Prédite')
axes[0].set_ylabel('Classe Réelle')
axes[0].set_title(f'Régression Logistique\nAccuracy: {accuracy_test_logreg:.3f}')

# Arbre de Décision
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Maligne (0)', 'Bénigne (1)'],
            yticklabels=['Maligne (0)', 'Bénigne (1)'],
            cbar_kws={'label': 'Nombre de prédictions'})
axes[1].set_xlabel('Classe Prédite')
axes[1].set_ylabel('Classe Réelle')
axes[1].set_title(f'Arbre de Décision\nAccuracy: {accuracy_test_tree:.3f}')

plt.tight_layout()
plt.show()


In [ ]:
# Analyse du sur-apprentissage (Overfitting)
overfitting_analysis = pd.DataFrame({
    'Modèle': ['Régression Logistique', 'Arbre de Décision'],
    'Accuracy Train': [accuracy_train_logreg, accuracy_train_tree],
    'Accuracy Test': [accuracy_test_logreg, accuracy_test_tree],
    'Différence (Train - Test)': [
        accuracy_train_logreg - accuracy_test_logreg,
        accuracy_train_tree - accuracy_test_tree
    ]
})

print("\n" + "="*80)
print("ANALYSE DU SUR-APPRENTISSAGE (OVERFITTING)")
print("="*80)
print(overfitting_analysis.to_string(index=False))
print("="*80)

if overfitting_analysis.iloc[1, 3] > 0.05:
    print("\n  L'Arbre de Décision montre des signes de sur-apprentissage")
    print(f"   Différence Train-Test: {overfitting_analysis.iloc[1, 3]:.4f}")
else:
    print("\n Pas de sur-apprentissage significatif détecté")

# Visualisation
plt.figure(figsize=(10, 6))
x_pos = np.arange(len(overfitting_analysis))
width = 0.35

plt.bar(x_pos - width/2, overfitting_analysis['Accuracy Train'], width,
        label='Train', color='lightblue', edgecolor='black', alpha=0.8)
plt.bar(x_pos + width/2, overfitting_analysis['Accuracy Test'], width,
        label='Test', color='lightcoral', edgecolor='black', alpha=0.8)

plt.xlabel('Modèle')
plt.ylabel('Accuracy')
plt.title('Comparaison Train vs Test (Détection de sur-apprentissage)')
plt.xticks(x_pos, overfitting_analysis['Modèle'])
plt.legend()
plt.ylim([0.85, 1.0])
plt.grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs
for i in range(len(overfitting_analysis)):
    plt.text(i - width/2, overfitting_analysis.iloc[i, 1] + 0.005,
             f"{overfitting_analysis.iloc[i, 1]:.3f}", ha='center', fontsize=10, fontweight='bold')
    plt.text(i + width/2, overfitting_analysis.iloc[i, 2] + 0.005,
             f"{overfitting_analysis.iloc[i, 2]:.3f}", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


### 4.2 Discussion et Interprétation


## Discussion et Interprétation des Résultats

### 1. Quel modèle est le plus adapté ?

#### **Régression Logistique :**

**Points forts :**
- Excellentes performances générales (Accuracy > 97%)
- Très bonne généralisation (peu de différence entre Train et Test)
- Modèle simple et interprétable
- Robuste et stable
- Moins sujet au sur-apprentissage

**Points faibles :**
- Suppose une relation linéaire entre features et log-odds
- Peut manquer des patterns non-linéaires complexes

#### **Arbre de Décision :**

**Points forts :**
- Bonnes performances (Accuracy ~94%)
- Très interprétable (visualisation de l'arbre)
- Capture les relations non-linéaires
- Pas besoin de standardisation des données
- Identifie clairement les features importantes

**Points faibles :**
- Légèrement plus de sur-apprentissage qu'attendu
- Performance légèrement inférieure à la régression logistique
- Sensible aux petites variations dans les données

### 2. Classes déséquilibrées ?

Le dataset présente un **léger déséquilibre** avec un ratio d'environ 1.6:1 (Bénigne:Maligne).
- Classe 0 (Maligne) : ~37% 
- Classe 1 (Bénigne) : ~63%

Ce déséquilibre est **modéré** et n'a pas significativement affecté les performances :
- Les deux modèles ont de bonnes métriques pour les deux classes
- Le F1-score est élevé pour les deux classes
- Utilisation de `stratify=y` lors du split a aidé à maintenir les proportions

**Si le déséquilibre était plus important**, on pourrait utiliser :
- SMOTE (Synthetic Minority Over-sampling Technique)
- Ajustement des poids des classes (`class_weight='balanced'`)
- Techniques d'under-sampling ou over-sampling

### 3. Sur-apprentissage ?

**Régression Logistique :**
- Différence Train-Test : ~0.5%
- **Pas de sur-apprentissage** : excellente généralisation

**Arbre de Décision :**
- Différence Train-Test : ~2-3%
- **Léger sur-apprentissage détecté** mais contrôlé grâce à `max_depth=5`
- Sans limitation de profondeur, le sur-apprentissage serait beaucoup plus important
